In [ ]:
# %% === Install Required Packages ===
!pip install --quiet pandas numpy matplotlib seaborn scikit-learn torch xgboost joblib tqdm pytorch-tabnet

In [ ]:
# %% === Imports ===
import pandas as pd, numpy as np, re, os, time, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import (
    RandomForestRegressor, GradientBoostingRegressor, 
    ExtraTreesRegressor, HistGradientBoostingRegressor
)
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import BayesianRidge 
import torch
import seaborn as sns
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import RidgeCV
from sklearn.neural_network import MLPRegressor
import xgboost as xgb
import joblib
from pytorch_tabnet.tab_model import TabNetRegressor
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

In [ ]:
# === Config ===
pd.options.mode.chained_assignment = None  # Suppress pandas SettingWithCopyWarning
torch.manual_seed(42)
np.random.seed(42)

DATASET_PATH = "../../../Dataset/Mohamed_Dataset/snapshots/youtube_dataset_snapshot_20251028_144628.csv"

In [ ]:
# %% === Load Data ===
data = pd.read_csv(DATASET_PATH, low_memory=False)
data['title'] = data['title'].astype(str)
data = data[~data['title'].str.contains('#', regex=False, na=False)] # Remove rows with '#' in title

In [ ]:
# %% === Date Features ===
data['publishedAt'] = pd.to_datetime(data['publishedAt'], errors='coerce')
data['pub_year'] = data['publishedAt'].dt.year
data['pub_month'] = data['publishedAt'].dt.month
data['pub_day'] = data['publishedAt'].dt.day
data['pub_weekday'] = data['publishedAt'].dt.weekday
data['pub_hour'] = data['publishedAt'].dt.hour
data['days_since_first_video'] = (data['publishedAt'] - data['publishedAt'].min()).dt.days

In [ ]:
# %% === Duration to Seconds ===
def duration_to_seconds(d):
    if not isinstance(d, str): return 0
    h = int(re.search(r'(\d+)H', d).group(1)) if 'H' in d else 0
    m = int(re.search(r'(\d+)M', d).group(1)) if 'M' in d else 0
    s = int(re.search(r'(\d+)S', d).group(1)) if 'S' in d else 0
    return h * 3600 + m * 60 + s

data['duration_sec'] = data['duration'].apply(duration_to_seconds)

In [ ]:
# %% === Clean & Filter ===
key_features = ['viewCount', 'likeCount', 'commentCount', 'pub_year']
data.dropna(subset=key_features, inplace=True)
data.drop_duplicates(subset='video_id', inplace=True, ignore_index=True)

# Keep only videos with reasonable views and published before the most recent day
data = data[
    (data['viewCount'] >= 1000) &
    (data['publishedAt'] <= data['publishedAt'].max() - pd.Timedelta(days=1))
]

In [ ]:
# %% === Feature Engineering, Splitting, and Preprocessing ===

# --- Basic feature creation ---
data['log_views'] = np.log1p(data['viewCount'])
data['title_length'] = data['title'].str.len().fillna(0)

# --- Split the dataframe first to prevent data leakage ---
train_df, temp_df = train_test_split(data, test_size=0.3, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

# --- TF-IDF Features for Title ---
tfidf_vectorizer = TfidfVectorizer(
    max_features=1000,  # top 1000 words
    ngram_range=(1,2),  # unigrams + bigrams
    stop_words='english'
)

# Fit on training titles only
tfidf_train = tfidf_vectorizer.fit_transform(train_df['title'])
tfidf_val = tfidf_vectorizer.transform(val_df['title'])
tfidf_test = tfidf_vectorizer.transform(test_df['title'])

# --- Channel Statistics (Calculated ONLY on Training Set) ---
if 'channel_id' in train_df.columns:
    ch_stats = train_df.groupby('channel_id')['viewCount'].agg(['mean','median','std','count']).fillna(0)
    ch_stats.columns = [f'channel_{c}_views' for c in ch_stats.columns]
    
    # Merge stats into each dataframe
    train_df = train_df.merge(ch_stats, on='channel_id', how='left')
    val_df   = val_df.merge(ch_stats, on='channel_id', how='left')
    test_df  = test_df.merge(ch_stats, on='channel_id', how='left')
    
    # Fill missing stats in val/test (channels not in train) with 0
    for col in ch_stats.columns:
        val_df[col] = val_df[col].fillna(0)
        test_df[col] = test_df[col].fillna(0)
else:
    for c in ['mean','median','std','count']:
        col_name = f'channel_{c}_views'
        train_df[col_name] = 0
        val_df[col_name] = 0
        test_df[col_name] = 0

# --- Categorical & Numeric Feature Preparation ---
numeric_features = [
    'duration_sec','pub_year','pub_month','pub_day','pub_weekday','pub_hour',
    'days_since_first_video',
    'channel_mean_views','channel_median_views','channel_std_views',
    'channel_count_views','title_length'
]
cat_cols = [c for c in ['definition','caption','category'] if c in data.columns]

# One-hot encode categorical features
train_df = pd.get_dummies(train_df, columns=cat_cols, drop_first=True)
val_df   = pd.get_dummies(val_df, columns=cat_cols, drop_first=True)
test_df  = pd.get_dummies(test_df, columns=cat_cols, drop_first=True)

# Align columns to ensure consistency
train_cols = train_df.columns
val_df = val_df.reindex(columns=train_cols, fill_value=0)
test_df = test_df.reindex(columns=train_cols, fill_value=0)

# --- Create final X and y sets ---
encoded_cat_cols = [c for c in train_df.columns if any(col in c for col in cat_cols)]
feature_cols = numeric_features + encoded_cat_cols

X_train = train_df[feature_cols].fillna(0)
y_train = train_df['log_views']

X_val = val_df[feature_cols].fillna(0)
y_val = val_df['log_views']

X_test = test_df[feature_cols].fillna(0)
y_test = test_df['log_views']

# --- Scale numeric features ---
scaler = StandardScaler()
X_train[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_val[numeric_features] = scaler.transform(X_val[numeric_features])
X_test[numeric_features] = scaler.transform(X_test[numeric_features])

# --- Combine numeric/categorical features with TF-IDF ---
from scipy.sparse import hstack

# Convert numeric features to float64 to avoid hstack errors
X_train_num = X_train.astype(np.float64).values
X_val_num   = X_val.astype(np.float64).values
X_test_num  = X_test.astype(np.float64).values

# Stack with TF-IDF (sparse)
X_train_full = hstack([X_train_num, tfidf_train])
X_val_full   = hstack([X_val_num, tfidf_val])
X_test_full  = hstack([X_test_num, tfidf_test])

# --- Optional: convert to dense for models that don't accept sparse input ---
X_train_full = X_train_full.toarray()
X_val_full   = X_val_full.toarray()
X_test_full  = X_test_full.toarray()

# --- Store actual views for later analysis ---
actuals = np.expm1(y_test)
actuals_val = np.expm1(y_val)

In [ ]:
# %% === Hyperparameter Tuning for RandomForestRegressor ===
from sklearn.model_selection import ParameterGrid

param_grid = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2']
}

grid = list(ParameterGrid(param_grid))

results = []
for params in tqdm(grid[:10], desc="Training RandomForest models"):
    model = RandomForestRegressor(random_state=42, n_jobs=-1, **params)
    
    t0 = time.perf_counter()
    model.fit(X_train_full, y_train)
    train_time = time.perf_counter() - t0
    
    preds_val_log = np.nan_to_num(model.predict(X_val_full), nan=0)
    preds_val = np.expm1(np.clip(preds_val_log, -20, 20))
    y_val_actual = np.expm1(y_val)
    
    val_mse = mean_squared_error(y_val_actual, preds_val)
    val_mae = mean_absolute_error(y_val_actual, preds_val)
    val_r2  = r2_score(y_val_actual, preds_val)
    median_error = np.median(np.abs((preds_val - y_val_actual) / (y_val_actual + 1e-9)) * 100)
    
    results.append({
        'params': params,
        'Val_MSE': val_mse,
        'Val_MAE': val_mae,
        'Val_R2': val_r2,
        'Median_Abs_Error (%)': median_error,
        'Train_Time': train_time
    })

res_df_val = pd.DataFrame(results).sort_values('Val_R2', ascending=False)
print(res_df_val)